In [1]:
import sys
import pickle

# TO CHANGE
BASEDIR = "../../"
sys.path.insert(0, BASEDIR)

In [15]:
from src.graph_main import RemoteKnowledgeGraph, RemoteKnowledgeGraphConfig
from src.knowledge_graph_model import GraphModelConfig, EmbeddingsModelConfig
from src.db_drivers.graph_driver import GraphDriverConfig, GraphDBConnectionConfig, DEFAULT_INMEMORYGRAPH_CONFIG
from src.db_drivers.vector_driver import VectorDriverConfig, EmbedderModelConfig, VectorDBConnectionConfig
from src.db_drivers.kv_driver import KeyValueDriverConfig, KVDBConnectionConfig, DEFAULT_INMEMORYKV_CONFIG

from src.qa_pipeline import QAPipelineConfig
from src.qa_pipeline.query_parser import QueryLLMParserConfig
from src.qa_pipeline.knowledge_comparator import KnowledgeComparatorConfig

from src.qa_pipeline.knowledge_retriever import KnowledgeRetrieverConfig
from src.qa_pipeline.knowledge_retriever.AStarTripletsRetriever import AStarGraphSearchConfig
from src.qa_pipeline.knowledge_retriever.BFSTripletsRetriever import BFSSearchConfig

from src.qa_pipeline.answer_generator import QALLMGeneratorConfig

from src.memorize_pipeline import MemPipelineConfig, LLMExtractorConfig, LLMUpdatorConfig

from src.utils import Logger, ReaderMetrics
from src.utils.data_structs import TripletCreator

#### 1. Загружем датасет с триплетами, на основе которого будет построен граф знаний

In [3]:
PKL_GRAPH_PATH = 'C:/Users/nikit/temp_files/pickled_graphs/DiaasqGigachat.pickle'

with open(PKL_GRAPH_PATH, 'rb') as f:
    formated_triplets = pickle.load(f)

In [4]:
print(len(formated_triplets))
formated_triplets = formated_triplets[:250]

211542


#### 2. Задаём конфигурацию графа знаний

In [7]:
inmemory_kg_config = RemoteKnowledgeGraphConfig(
    
    graph_struct_config=GraphModelConfig(driver_config=GraphDriverConfig(
        db_vendor='inmemory_graph', db_config=DEFAULT_INMEMORYGRAPH_CONFIG)), # TO CHANGE
    
    embedds_struct_config=EmbeddingsModelConfig(
        nodesdb_driver_config=VectorDriverConfig(db_vendor='chroma', db_config=VectorDBConnectionConfig(
            path='../../data/graph_structures/vectorized_nodes/testing6', db_name='vectorized_nodes', is_exist=True, need_to_clear=True)), # TO CHANGE
        tripletsdb_driver_config=VectorDriverConfig(db_vendor='chroma', db_config=VectorDBConnectionConfig(
            path='../../data/graph_structures/vectorized_triplets/testing6', db_name='vectorized_triplets', is_exist=True, need_to_clear=True)), # TO CHANGE
        embedder_config=EmbedderModelConfig(model_name_or_path='intfloat/multilingual-e5-small')),
    
    qa_pipeline_config=QAPipelineConfig(
        query_parser_config=QueryLLMParserConfig(),
        knowledge_comparator_config=KnowledgeComparatorConfig(),
        knowledge_retriever_config=KnowledgeRetrieverConfig(
            retriever_method='astar', # TO CHANGE
            retriever_config=AStarGraphSearchConfig(), # TO CHANGE
            cache_config=KeyValueDriverConfig(db_vendor='inmemory_kv', db_config=DEFAULT_INMEMORYKV_CONFIG)), # TO CHANGE
        answer_generator_config=QALLMGeneratorConfig()),
    
    mem_pipeline_config=MemPipelineConfig(
        extractor_config=LLMExtractorConfig(),
        updator_config=LLMUpdatorConfig()),
    
    log=Logger('log/main'))

#### 3. Инициализируем граф знаний

In [8]:
rkg_main = RemoteKnowledgeGraph(config=inmemory_kg_config)

c:\Users\nikit\anaconda3\envs\LLM\Lib\site-packages\huggingface_hub\file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [9]:
# ATTENTION !!!
#rkg_main.kg_model.graph_struct.db_conn.execute_query("match (a) -[r] -> () delete a, r")
#rkg_main.kg_model.graph_struct.db_conn.execute_query("match (a) delete a")
# ATTENTION !!!

#### 4. Добавляем в граф загруженные триплеты

In [10]:
rkg_main.kg_model.graph_struct.create_triplets(formated_triplets)
rkg_main.kg_model.embeddings_struct.add_triplets(formated_triplets)

100%|██████████| 2/2 [00:00<00:00,  3.48it/s]


#### 5. Q&A

In [19]:
for triplet in formated_triplets[:250]:
    print(TripletCreator.stringify(triplet)[1])

4.9.2019: amanda compares xiaomi mi 12pro
4.9.2019: gregory claims iqoo9 better heat dissipation
4.9.2019: jordan says similar hot plate area
4.9.2019: Gregory uses IQOO9
4.9.2019: Gregory uses IQOO9
4.9.2019: 
Alan mentions digital bloggers
4.9.2019: 
Alan mentions digital bloggers
4.9.2019: 
Kyle talks about cost-effective machines
4.9.2019: 
Kyle talks about cost-effective machines
4.9.2019: gregory uses iqoo9
4.9.2019: alan asks does everyone need cost-effective machine
4.9.2019: kyle mentions hardware shrink
4.9.2019: deborah suggests trying in summer
4.9.2019: gregory mentions fever in summer
4.9.2019: ann says stable high frame rate
4.9.2019: 
Alan says sub-brands are cost-effective
4.9.2019: 
Alan says sub-brands are cost-effective
4.9.2019: 
Alan says sub-brands are cost-effective
4.9.2019: 
Deborah suggests trying IQOO9 in summer
4.9.2019: 
Deborah suggests trying IQOO9 in summer
4.9.2019: 
Deborah suggests trying IQOO9 in summer
4.9.2019: 
Gregory mentions 865, 870, and A13 

: 

In [ ]:
examples_questions = []

In [ ]:
rkg_main.answer_question(examples_questions[0])